# Comparative Sentiment Analysis: Indonesian E-Commerce Reviews (Scopus Pipeline)
## Notebook 04: IndoBERT Fine-Tuning with Class-Weighted Loss & Extended Context (Length 256)

```text
Author: Muhammad Farrel Haidar
Project: AI PR & KOL Specialist DSS
Date: 2026-10-14
Study: Comparative Sentiment Analysis (IndoBERT vs Tree-Based Models)
Model: indobenchmark/indobert-base-p1 (Transformer Architecture, 110M Parameters)
Optimization: Class-Weighted Cross-Entropy Loss, max_length=256, Warmup 10%, Early Stopping (Patience=2)
Hardware Acceleration: NVIDIA GeForce RTX 4070 (8GB VRAM) / Google Colab (Tesla T4 / A100)
Test Performance: Accuracy 65.26%, Macro-F1 61.76%, ROC-AUC 0.8055
```

---
### 1. Landasan Teoretis & Formulasi Matematis Transformer (Pilar 2)

#### a. Mekanisme Multi-Head Scaled Dot-Product Self-Attention
Inti arsitektur IndoBERT adalah mekanisme atensi kontekstual bidirectional yang memetakan keterkaitan semantik antar-kata secara dinamis:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

di mana $Q$ (Query), $K$ (Key), dan $V$ (Value) adalah proyeksi linear representasi teks berdimensi $d_k$.

#### b. Fungsi Objektif: Class-Weighted Cross-Entropy Loss
Untuk mengatasi fenomena *Neutral Collapse* tanpa mengorbankan kelas sentimen lain, kami memformulasikan *Class-Weighted Cross-Entropy Loss*:

$$\mathcal{L}_{\text{WCE}} = -\sum_{i=1}^{N} w_{y_i} \sum_{c=0}^{2} y_{i,c} \log(\hat{y}_{i,c})$$

di mana bobot kelas dihitung secara invers terhadap frekuensi kelas:

$$w_c = \frac{N}{|C| \times N_c}$$

sehingga model terdorong untuk membedakan batas keputusan pada kelas minoritas/netral secara presisi.

In [ ]:
# Author: Muhammad Farrel Haidar
# Project: AI PR & KOL Specialist DSS
# Date: 2026-10-14

# 1. Environment & GPU Hardware Detection (Colab / Local Auto-Detect)
import os
import sys
import time
import torch

# Google Colab Mount Setup (Jika dijalankan di Colab)
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Mounted Google Drive successfully.")
    !pip install -q transformers==4.44.2 datasets==2.21.0 evaluate accelerate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB)  : {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

### 2. Import Libraries & Dataset Tokenization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

set_seed(42)

# Load Split Data
data_dir = "../data/splits" if os.path.exists("../data/splits") else "data/splits"
train_df = pd.read_csv(f"{data_dir}/train.csv")
val_df = pd.read_csv(f"{data_dir}/val.csv")
test_df = pd.read_csv(f"{data_dir}/test.csv")

print(f"Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")

# Hitung Class Weights untuk Weighted Loss (Pilar 2)
classes = np.array([0, 1, 2])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["label"].values)
class_weights_tensor = torch.tensor(weights, dtype=torch.float)
print(f"Class Weights: Negatif={weights[0]:.4f}, Netral={weights[1]:.4f}, Positif={weights[2]:.4f}")

MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = int(self.labels[item])
        encoding = self.tokenizer(text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_ds = ReviewDataset(train_df["cleaned_text"].fillna(""), train_df["label"].values, tokenizer, max_len=256)
val_ds = ReviewDataset(val_df["cleaned_text"].fillna(""), val_df["label"].values, tokenizer, max_len=256)
test_ds = ReviewDataset(test_df["cleaned_text"].fillna(""), test_df["label"].values, tokenizer, max_len=256)
print("Datasets initialized with max_length=256.")

### 3. Hyperparameter Configuration & Custom Compute Metrics

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is not None:
            weights = self.class_weights.to(model.device)
            loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

output_dir = "models/indobert_finetuned"
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    warmup_ratio=0.10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=42,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
print("WeightedTrainer initialized with Early Stopping (patience=2) & Warmup 10%.")

### 4. Pelatihan Model & Evaluasi Test Set

In [ ]:
# Load hasil evaluasi checkpoint model yang telah dilatih
res_csv = "../data/results/metrics_indobert.csv" if os.path.exists("../data/results/metrics_indobert.csv") else "data/results/metrics_indobert.csv"
if os.path.exists(res_csv):
    m_indobert = pd.read_csv(res_csv)
    print("=== METRIK HASIL EVALUASI TEST SET INDOBERT (OPSI B) ===")
    display(m_indobert)
else:
    t0 = time.time()
    trainer.train()
    print(f"Training finished in {time.time()-t0:.2f}s")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    preds = trainer.predict(test_ds)
    y_pred = np.argmax(preds.predictions, axis=-1)
    y_true = test_df["label"].values
    print(f"Test Accuracy: {accuracy_score(y_true, y_pred):.4f}")

### 5. Visualisasi Confusion Matrix & Kurva Konvergensi

In [ ]:
from PIL import Image
fig3_path = "../paper_assets/figure3_training_curves_indobert.png" if os.path.exists("../paper_assets") else "paper_assets/figure3_training_curves_indobert.png"
fig4_path = "../paper_assets/figure4_confusion_matrix_indobert.png" if os.path.exists("../paper_assets") else "paper_assets/figure4_confusion_matrix_indobert.png"

if os.path.exists(fig3_path):
    plt.figure(figsize=(10, 5), dpi=200)
    plt.imshow(Image.open(fig3_path))
    plt.axis("off")
    plt.title("Figure 3: Training Loss & Validation Macro-F1 Convergence Curves", pad=12)
    plt.show()

if os.path.exists(fig4_path):
    plt.figure(figsize=(6, 5), dpi=200)
    plt.imshow(Image.open(fig4_path))
    plt.axis("off")
    plt.title("Figure 4: Confusion Matrix IndoBERT Fine-Tuned (Test Set)", pad=12)
    plt.show()